In [1]:
import pandas as pd
import numpy as np


In [5]:

orders = pd.read_csv('/olist_orders_dataset.csv')
order_items = pd.read_csv('/olist_order_items_dataset.csv')
order_reviews = pd.read_csv('/olist_order_reviews_dataset.csv')
order_payments = pd.read_csv('/olist_order_payments_dataset.csv')
customers = pd.read_csv('/olist_customers_dataset.csv')
products = pd.read_csv('/olist_products_dataset.csv')
sellers = pd.read_csv('/olist_sellers_dataset.csv')

print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("order_reviews:", order_reviews.shape)
print("order_payments:", order_payments.shape)
print("customers:", customers.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)


orders: (99441, 8)
order_items: (112650, 7)
order_reviews: (99224, 7)
order_payments: (103886, 5)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)


In [6]:
print("orders columns:")
print(orders.columns.tolist())
print()

print("order_items columns:")
print(order_items.columns.tolist())
print()

print("order_reviews columns:")
print(order_reviews.columns.tolist())
print()

print("customers columns:")
print(customers.columns.tolist())
print()

print("products columns:")
print(products.columns.tolist())
print()

print("sellers columns:")
print(sellers.columns.tolist())


orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

order_items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

order_reviews columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

customers columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

products columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

sellers columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [7]:
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
order_reviews = order_reviews.drop_duplicates()

orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], errors='coerce'
)

order_reviews['review_creation_date'] = pd.to_datetime(
    order_reviews['review_creation_date'], errors='coerce'
)

print("Cleaning completed")


Cleaning completed


In [8]:
print(orders.isnull().sum())
print()
print(customers.isnull().sum())
print()
print(products.isnull().sum())
print()
print(sellers.isnull().sum())


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


In [19]:
orders['order_estimated_delivery_date'] = pd.to_datetime(
    orders['order_estimated_delivery_date'], errors='coerce'
)

orders['delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

print(orders[['order_id', 'delivery_days', 'delay_days']].head())


                           order_id  delivery_days  delay_days
0  e481f51cbdc54678b7cc49136f2d6af7            8.0        -8.0
1  53cdb2fc8bc7dce0b6741e2150273451           13.0        -6.0
2  47770eb9100c2d0c44946d9cf07ec65d            9.0       -18.0
3  949d5b44dbf5de918fe9c16f97b45f8a           13.0       -13.0
4  ad21c59c0840e6cb83a9ceb5573f8159            2.0       -10.0


In [10]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'], errors='coerce')

orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

print(orders[['order_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_days']].head())


                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49   
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:06   
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:39   

  order_delivered_customer_date  delivery_days  
0           2017-10-10 21:25:13            8.0  
1           2018-08-07 15:27:45           13.0  
2           2018-08-17 18:06:29            9.0  
3           2017-12-02 00:28:42           13.0  
4           2018-02-16 18:17:02            2.0  


In [11]:
dim_customers = customers[[
    'customer_id',
    'customer_unique_id',
    'customer_city',
    'customer_state'
]].copy()

dim_customers = dim_customers.drop_duplicates().reset_index(drop=True)
dim_customers.insert(0, 'customer_key', range(1, len(dim_customers) + 1))

print(dim_customers.head())
print(dim_customers.shape)


   customer_key                       customer_id  \
0             1  06b8999e2fba1a1fbc88172c00ba8bc7   
1             2  18955e83d337fd6b2def6b18a428ac77   
2             3  4e7b3e00288586ebd08712fdd0374a03   
3             4  b2b6027bc5c5109e529d4dc6358b12c3   
4             5  4f2d8ab171c80ec8364f7c12e35b23ad   

                 customer_unique_id          customer_city customer_state  
0  861eff4711a542e4b93843c6dd7febb0                 franca             SP  
1  290c77bc529b7ac935b93aa66c333dc3  sao bernardo do campo             SP  
2  060e732b5b29e8181a18229c7b0b2b5e              sao paulo             SP  
3  259dac757896d24d7702b9acbbff3f3c        mogi das cruzes             SP  
4  345ecd01c38d18a9036ed96c73b8d066               campinas             SP  
(99441, 5)


In [12]:
dim_products = products[[
    'product_id',
    'product_category_name',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]].copy()

dim_products = dim_products.drop_duplicates().reset_index(drop=True)
dim_products.insert(0, 'product_key', range(1, len(dim_products) + 1))

print(dim_products.head())
print(dim_products.shape)


   product_key                        product_id  product_category_name  \
0            1  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1            2  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2            3  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3            4  cef67bcfe19066a932b7673e239eb23d                  bebes   
4            5  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0       

In [13]:
dim_sellers = sellers[[
    'seller_id',
    'seller_city',
    'seller_state'
]].copy()

dim_sellers = dim_sellers.drop_duplicates().reset_index(drop=True)
dim_sellers.insert(0, 'seller_key', range(1, len(dim_sellers) + 1))

print(dim_sellers.head())
print(dim_sellers.shape)


   seller_key                         seller_id        seller_city  \
0           1  3442f8959a84dea7ee197c632cb2df15           campinas   
1           2  d1b65fc7debc3361ea86b5f14c68d2e2         mogi guacu   
2           3  ce3ad9de960102d0677a81f5d0bb7b2d     rio de janeiro   
3           4  c0f3eea2e14555b6faeea3dd58c1b1c3          sao paulo   
4           5  51a04a8a6bdcb23deccc82b0b80742cf  braganca paulista   

  seller_state  
0           SP  
1           SP  
2           RJ  
3           SP  
4           SP  
(3095, 4)


In [14]:
purchase_dates = orders[['order_purchase_timestamp']].copy()
purchase_dates.columns = ['full_date']

review_dates = order_reviews[['review_creation_date']].copy()
review_dates.columns = ['full_date']

all_dates = pd.concat([purchase_dates, review_dates], ignore_index=True)
all_dates = all_dates.dropna().drop_duplicates().sort_values('full_date').reset_index(drop=True)

dim_date = all_dates.copy()
dim_date['full_date'] = pd.to_datetime(dim_date['full_date']).dt.date

dim_date['year'] = pd.to_datetime(dim_date['full_date']).dt.year
dim_date['quarter'] = pd.to_datetime(dim_date['full_date']).dt.quarter
dim_date['month'] = pd.to_datetime(dim_date['full_date']).dt.month
dim_date['month_name'] = pd.to_datetime(dim_date['full_date']).dt.month_name()
dim_date['day'] = pd.to_datetime(dim_date['full_date']).dt.day

dim_date = dim_date.drop_duplicates().reset_index(drop=True)
dim_date.insert(0, 'date_key', range(1, len(dim_date) + 1))

print(dim_date.head())
print(dim_date.shape)


   date_key   full_date  year  quarter  month month_name  day
0         1  2016-09-04  2016        3      9  September    4
1         2  2016-09-05  2016        3      9  September    5
2         3  2016-09-13  2016        3      9  September   13
3         4  2016-09-15  2016        3      9  September   15
4         5  2016-10-02  2016        4     10    October    2
(683, 7)


In [20]:
fact_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp', 'order_status', 'delivery_days', 'delay_days']],
    on='order_id',
    how='left'
)

fact_sales['order_purchase_date'] = pd.to_datetime(
    fact_sales['order_purchase_timestamp'], errors='coerce'
).dt.date

fact_sales = fact_sales.merge(
    dim_customers[['customer_key', 'customer_id']],
    on='customer_id',
    how='left'
)

fact_sales = fact_sales.merge(
    dim_products[['product_key', 'product_id']],
    on='product_id',
    how='left'
)

fact_sales = fact_sales.merge(
    dim_sellers[['seller_key', 'seller_id']],
    on='seller_id',
    how='left'
)

fact_sales = fact_sales.merge(
    dim_date[['date_key', 'full_date']],
    left_on='order_purchase_date',
    right_on='full_date',
    how='left'
)

fact_sales['total_value'] = fact_sales['price'] + fact_sales['freight_value']

fact_sales = fact_sales[[
    'order_id',
    'order_item_id',
    'date_key',
    'customer_key',
    'product_key',
    'seller_key',
    'order_status',
    'price',
    'freight_value',
    'total_value',
    'delivery_days',
    'delay_days'
]]

fact_sales = fact_sales.rename(columns={'date_key': 'order_date_key'})

fact_sales = fact_sales[fact_sales['price'] > 0]
fact_sales = fact_sales.dropna(subset=['order_date_key', 'customer_key', 'product_key', 'seller_key'])

fact_sales['order_date_key'] = fact_sales['order_date_key'].astype(int)
fact_sales['customer_key'] = fact_sales['customer_key'].astype(int)
fact_sales['product_key'] = fact_sales['product_key'].astype(int)
fact_sales['seller_key'] = fact_sales['seller_key'].astype(int)

print(fact_sales.head())
print(fact_sales.shape)


                           order_id  order_item_id  order_date_key  \
0  00010242fe8c5a6d1ba2dd792cb16214              1             316   
1  00018f77f2f0320c557190d7a144bdd3              1             176   
2  000229ec398224ef6ca0657da4fc703e              1             439   
3  00024acbcdf0a6daa1e931b038114c75              1             645   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1              95   

   customer_key  product_key  seller_key order_status   price  freight_value  \
0         65558        25866         514    delivered   58.90          13.29   
1         34266        27231         472    delivered  239.90          19.93   
2         34956        22625        1825    delivered  199.00          17.87   
3         51764        15404        2024    delivered   12.99          12.79   
4          7603         8863        1598    delivered  199.90          18.14   

   total_value  delivery_days  delay_days  
0        72.19            7.0        -9.0  
1       25

In [21]:
fact_reviews = order_reviews.merge(
    orders[['order_id', 'customer_id', 'order_status', 'delivery_days', 'delay_days']],
    on='order_id',
    how='left'
)

fact_reviews['review_creation_date'] = pd.to_datetime(
    fact_reviews['review_creation_date'], errors='coerce'
).dt.date

fact_reviews = fact_reviews.merge(
    dim_customers[['customer_key', 'customer_id']],
    on='customer_id',
    how='left'
)

fact_reviews = fact_reviews.merge(
    dim_date[['date_key', 'full_date']],
    left_on='review_creation_date',
    right_on='full_date',
    how='left'
)

fact_reviews = fact_reviews[[
    'review_id',
    'order_id',
    'date_key',
    'customer_key',
    'review_score',
    'order_status',
    'delivery_days',
    'delay_days'
]]

fact_reviews = fact_reviews.rename(columns={'date_key': 'review_date_key'})

fact_reviews = fact_reviews.dropna(subset=['review_date_key', 'customer_key', 'review_score'])

fact_reviews['review_date_key'] = fact_reviews['review_date_key'].astype(int)
fact_reviews['customer_key'] = fact_reviews['customer_key'].astype(int)
fact_reviews['review_score'] = fact_reviews['review_score'].astype(int)

print(fact_reviews.head())
print(fact_reviews.shape)


                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_date_key  customer_key  review_score order_status  delivery_days  \
0              443         44071             4    delivered            6.0   
1              494         92399             5    delivered            9.0   
2              473         30603             5    delivered           13.0   
3              171         70156             5    delivered           10.0   
4              485         64935             5    delivered           18.0   

   delay_days  
0       -16.0  
1        -5.0  
2       -21.0  
3       -20.0  
4        -9.

In [22]:
print("dim_customers:", dim_customers.shape)
print("dim_products:", dim_products.shape)
print("dim_sellers:", dim_sellers.shape)
print("dim_date:", dim_date.shape)
print("fact_sales:", fact_sales.shape)
print("fact_reviews:", fact_reviews.shape)

print()
print("Missing customer_key in fact_sales:", fact_sales['customer_key'].isnull().sum())
print("Missing product_key in fact_sales:", fact_sales['product_key'].isnull().sum())
print("Missing seller_key in fact_sales:", fact_sales['seller_key'].isnull().sum())
print("Missing order_date_key in fact_sales:", fact_sales['order_date_key'].isnull().sum())

print()
print("Missing customer_key in fact_reviews:", fact_reviews['customer_key'].isnull().sum())
print("Missing review_date_key in fact_reviews:", fact_reviews['review_date_key'].isnull().sum())
print("Missing review_score in fact_reviews:", fact_reviews['review_score'].isnull().sum())


dim_customers: (99441, 5)
dim_products: (32951, 10)
dim_sellers: (3095, 4)
dim_date: (683, 7)
fact_sales: (112650, 12)
fact_reviews: (99224, 8)

Missing customer_key in fact_sales: 0
Missing product_key in fact_sales: 0
Missing seller_key in fact_sales: 0
Missing order_date_key in fact_sales: 0

Missing customer_key in fact_reviews: 0
Missing review_date_key in fact_reviews: 0
Missing review_score in fact_reviews: 0


In [23]:
dim_customers.to_csv('dim_customers.csv', index=False)
dim_products.to_csv('dim_products.csv', index=False)
dim_sellers.to_csv('dim_sellers.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
fact_sales.to_csv('fact_sales.csv', index=False)
fact_reviews.to_csv('fact_reviews.csv', index=False)

print("Final warehouse files exported successfully")


Final warehouse files exported successfully


In [24]:
print(dim_date.head())
print()
print(fact_sales.head())
print()
print(fact_reviews.head())


   date_key   full_date  year  quarter  month month_name  day
0         1  2016-09-04  2016        3      9  September    4
1         2  2016-09-05  2016        3      9  September    5
2         3  2016-09-13  2016        3      9  September   13
3         4  2016-09-15  2016        3      9  September   15
4         5  2016-10-02  2016        4     10    October    2

                           order_id  order_item_id  order_date_key  \
0  00010242fe8c5a6d1ba2dd792cb16214              1             316   
1  00018f77f2f0320c557190d7a144bdd3              1             176   
2  000229ec398224ef6ca0657da4fc703e              1             439   
3  00024acbcdf0a6daa1e931b038114c75              1             645   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1              95   

   customer_key  product_key  seller_key order_status   price  freight_value  \
0         65558        25866         514    delivered   58.90          13.29   
1         34266        27231         472    de